# Coaching Agent — Demo Notebook

**Stack:** gemma3:4b (Ollama) · ChromaDB · HuggingFace Embeddings · LangChain

**Pipeline:**
```
PatientContext → RAG Retrieval → gemma3:4b → Polish → CoachingOutput
```

---
### ⚠️ Before running: make sure Ollama is running
```bash
ollama serve
ollama pull gemma3:4b
```

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '.')  # ensure local modules are importable

from coaching_agent import CoachingAgent
from rag_retriever import CoachingKnowledgeBase
from schemas import PatientContext, RehabPhase, ConditionCategory, ExerciseRecord, make_sample_context

print('All modules loaded ✓')

## 2. Build / Load Knowledge Base

First run: indexes `dataset/clean` → saves to `./chroma_coaching_db`

Subsequent runs: loads from disk (fast)

In [ ]:
kb = CoachingKnowledgeBase(
    data_dir='dataset/clean',        # ← your data directory
    persist_dir='./chroma_coaching_db',
    chunk_size=500,
    chunk_overlap=50,
).load_or_build(
    force_rebuild=False              # set True to re-index from scratch
)

## 3. Initialise Agent

In [ ]:
agent = CoachingAgent(
    knowledge_base=kb,
    ollama_model='gemma3:4b',
    retrieval_k=3,        # Keep at 3 to avoid memory pressure with 4b model
    enable_polish=True,   # Set False to skip 2nd LLM call (faster, less memory)
    verbose=True,
)

## 4. Run — Sample Scenarios

### Scenario A: Knee Osteoarthritis (Week 10)

In [ ]:
context_knee = make_sample_context('knee')
print('Patient context:')
print(f'  Condition: {context_knee.condition}')
print(f'  Phase: {context_knee.rehab_phase.value}')
print(f'  Message: "{context_knee.patient_message}"')

In [ ]:
output = agent.generate_coaching(context_knee)

print('='*60)
print('COACHING FEEDBACK')
print('='*60)
print(output.coaching_feedback)
print()
print(f'Sources: {output.retrieved_sources}')
print(f'Confidence: {output.confidence_score:.0%}')

### Scenario B: Shoulder Rehab (Week 3)

In [ ]:
output_shoulder = agent.generate_coaching(make_sample_context('shoulder'))
print(output_shoulder.coaching_feedback)

### Scenario C: ACL Recovery (Week 20 — Return to Sport)

In [ ]:
output_acl = agent.generate_coaching(make_sample_context('acl'))
print(output_acl.coaching_feedback)

## 5. Custom Patient Input

Build your own PatientContext for testing:

In [ ]:
custom_context = PatientContext(
    patient_id='P999',
    condition='patellofemoral pain syndrome',
    condition_category=ConditionCategory.KNEE,
    rehab_phase=RehabPhase.EARLY,
    pain_level=6,
    weeks_into_rehab=4,
    recent_exercises=[
        ExerciseRecord('Clamshells', sets=3, reps=15, completed=True, difficulty_feedback='ok'),
        ExerciseRecord('Wall sits', sets=2, reps=None, completed=False, difficulty_feedback='too hard'),
    ],
    patient_message='The wall sits really hurt behind my kneecap. Should I stop doing them?',
    age=32,
    goals='Run a 5K in 3 months',
)

custom_output = agent.generate_coaching(custom_context)
print(custom_output.coaching_feedback)

## 6. Inspect Structured Output

In [ ]:
# Look at the parsed structured fields
print('📋 Suggested exercises:')
for ex in custom_output.suggested_exercises:
    print(f'  • {ex}')

print('\n⚠️ Safety notes:')
for note in custom_output.safety_notes:
    print(f'  {note}')

print(f'\n💪 Closing motivation:')
print(f'  {custom_output.motivational_note}')

print(f'\n📚 Retrieved from: {custom_output.retrieved_sources}')
print(f'🎯 Confidence score: {custom_output.confidence_score:.0%}')

## 7. Memory-Safe Tips

If you see **kernel crashes** (like in the original trial.ipynb):

1. **Disable polish** — set `enable_polish=False` in CoachingAgent
2. **Reduce retrieval** — set `retrieval_k=2`
3. **Run one scenario at a time** — restart kernel between heavy runs
4. **Check Ollama memory** — `ollama ps` to see VRAM usage
5. **Use `num_predict=300`** — edit coaching_agent.py line ~45